In [1]:
import jax
print(jax.devices())
print(jax.local_device_count())

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]
1


In [3]:
import jax
import jax.numpy as jnp
import time

def matmul(x, y):
    return jnp.dot(x, y)

jit_matmul = jax.jit(matmul)

x = jnp.ones((4096, 4096))
y = jnp.ones((4096, 4096))

# Eager
t0 = time.time()
matmul(x, y).block_until_ready()
print("Eager:", round(time.time()-t0, 4), "sec")

# JIT warm-up (compilation happens here)
t0 = time.time()
jit_matmul(x, y).block_until_ready()
print("JIT warm-up:", round(time.time()-t0, 4), "sec")

# JIT cached (pure execution)
t0 = time.time()
jit_matmul(x, y).block_until_ready()
print("JIT cached:", round(time.time()-t0, 4), "sec")

# v5e-1 has 1 core, so we simulate with vmap instead of pmap
# This is the correct approach for single-core TPU

def per_device_sum(x):
    return jnp.sum(x)

# vmap vectorizes across batch dimension
batched_sum = jax.vmap(per_device_sum)

data = jnp.ones((4, 1024, 1024))  # 4 independent chunks
result = batched_sum(data)
print("Output shape:", result.shape)
print("Values:", result)

def matmul(x, y):
    return jnp.dot(x, y)

x = jnp.ones((4096, 4096))
y = jnp.ones((4096, 4096))

print(jax.make_jaxpr(matmul)(x, y))

Eager: 0.0012 sec
JIT warm-up: 0.5717 sec
JIT cached: 0.0011 sec
Output shape: (4,)
Values: [1048576. 1048576. 1048576. 1048576.]
{ lambda ; a:f32[4096,4096] b:f32[4096,4096]. let
    c:f32[4096,4096] = dot_general[
      dimension_numbers=(([1], [0]), ([], []))
      preferred_element_type=float32
    ] a b
  in (c,) }
